# 第10回講義 宿題



## 課題
自己教師あり学習を用いて事前学習を行い，得られた表現をLinear probingで評価してみましょう．  
ネットワークの形などに制限はとくになく，今回のLessonで扱った内容以外の工夫も組み込んでもらって構いません．   



### 目標値
なし
- 自己教師あり学習の手法によっては計算リソースによって性能が大きく変わるため，目標精度は設定していません．
- ただし以下の工夫を行うことで計算リソースが少なくとも，長い学習を分割して行うことができます．  
    - model，optimizer, schedulerを一定エポックで保存して，読み込むことで学習を再開することができます．
    - 演習のようにschedulerを実装した場合は保存は必要なく，同じ引数でインスタンスを作成して`__call__`の際に与えるepochを学習の続きから与えれば動作します．  
    - 参考: https://pytorch.org/tutorials/beginner/saving_loading_models.html



### ルール
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- 自己教師あり学習では以下のセルで指定されている`x_train`以外の学習データは用いないでください．
- Linear probingの際には`x_train`, `t_train`以外の学習データは用いないでください．


### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルを`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第10回 表現学習と自己教師あり学習」を選択して**提出してください．
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第10回 表現学習と自己教師あり学習 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．


### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 作業ディレクトリを指定
work_dir = 'drive/MyDrive/DLBasic/HW/HW10'

### データの読み込み（このセルは修正しないでください）

In [3]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm.notebook import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import math
import os
from einops import rearrange, repeat
from einops.layers.torch import Rearrange


#学習データ
x_train = np.load(work_dir + '/Lecture10/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture10/data/t_train.npy')

#テストデータ
x_test = np.load(work_dir + '/Lecture10/data/x_test.npy')

class train_dataset(torch.utils.data.Dataset):
    def __init__(self, x_train, t_train):
        data = x_train.astype('float32')
        self.x_train = []
        for i in range(data.shape[0]):
            self.x_train.append(Image.fromarray(np.uint8(data[i])))
        self.t_train = t_train
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_train)

    def __getitem__(self, idx):
        return self.transform(self.x_train[idx]), torch.tensor(t_train[idx], dtype=torch.long)

class test_dataset(torch.utils.data.Dataset):
    def __init__(self, x_test):
        data = x_test.astype('float32')
        self.x_test = []
        for i in range(data.shape[0]):
            self.x_test.append(Image.fromarray(np.uint8(data[i])))
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_test)

    def __getitem__(self, idx):
        return self.transform(self.x_test[idx])

trainval_data = train_dataset(x_train, t_train)
test_data = test_dataset(x_test)

### データローダの準備  

In [13]:
val_size = 3000
train_indices, val_indices = torch.utils.data.random_split(range(len(trainval_data)), [len(trainval_data) - val_size, val_size])

train_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.RandomHorizontalFlip(),
     transforms.RandomResizedCrop(32, scale=(0.5, 1.0)),
     transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
     transforms.Normalize(0.5, 0.5)]
)
test_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(0.5, 0.5)]
)

train_data = torch.utils.data.Subset(train_dataset(x_train, t_train), train_indices)
train_data.dataset.transform = train_transform
valid_data = torch.utils.data.Subset(train_dataset(x_train, t_train), val_indices)
valid_data.dataset.transform = test_transform

test_data.transform = test_transform

batch_size = 128

dataloader_train = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

dataloader_valid = torch.utils.data.DataLoader(
    valid_data,
    batch_size=batch_size,
    shuffle=False
)

dataloader_test = torch.utils.data.DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

### 自己教師あり学習の実装
MAEを利用することを想定していますが，他の自己教師あり学習を利用していただいて構いません．   

In [5]:
def fix_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


fix_seed(seed=42)


def random_indexes(size):
    """
    パッチをランダムに並べ替えるためのindexを生成する関数．

    Argument
    --------
    size : int
        入力されるパッチの数（系列長Nと同じ値）．
    """
    forward_indexes = np.arange(size)  # 0からsizeまでを並べた配列を作成
    np.random.shuffle(forward_indexes)  # 生成した配列をシャッフルすることで，パッチの順番をランダムに決定
    backward_indexes = np.argsort(forward_indexes)  # 並べ替えたパッチをもとの順番に戻すためのidx

    return forward_indexes, backward_indexes


def take_indexes(sequences, indexes):
    """
    パッチを並べ替えるための関数．

    Argument
    --------
    sequences : torch.Tensor
        入力画像をパッチ分割したデータ．(B, N, dim)の形状をしている．
    indexes : np.ndarray
        並べ替えるために利用するindex．
        random_indexesで生成したforward_indexesかbackward_indexesが入ることが想定されている．
    """
    return torch.gather(sequences, dim=1, index=indexes.unsqueeze(2).repeat(1, 1, sequences.shape[-1]))


class Attention(nn.Module):
    def __init__(self, dim, heads, dim_head, dropout=0.):
        super().__init__()
        self.dim = dim
        self.dim_head = dim_head
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = math.sqrt(dim_head)

        self.attend = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

        # Q, K, Vに変換するための全結合層
        self.to_q = nn.Linear(in_features=dim, out_features=inner_dim)
        self.to_k = nn.Linear(in_features=dim, out_features=inner_dim)
        self.to_v = nn.Linear(in_features=dim, out_features=inner_dim)

        # dim != inner_dimなら線形層を入れる，そうでなければそのまま出力
        self.to_out = nn.Sequential(
            nn.Linear(in_features=inner_dim, out_features=dim),
            nn.Dropout(dropout),
        ) if project_out else nn.Identity()

    def forward(self, x):
        """
        B: バッチサイズ
        N: 系列長
        D: データの次元数(dim)
        """
        B, N, D = x.size()

        # 入力データをQ, K, Vに変換する
        # (B, N, dim) -> (B, N, inner_dim)
        q = self.to_q(x)
        k = self.to_k(x)
        v = self.to_v(x)

        # Q, K, Vをヘッドに分割する
        # (B, N, inner_dim) -> (B, heads, N, dim_head)
        q = rearrange(q, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)
        k = rearrange(k, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)
        v = rearrange(v, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)

        # QK^T / sqrt(d_k)を計算する
        # (B, heads, N, dim_head) x (B, heads, dim_head, N) -> (B, heads, N, N)
        dots = torch.matmul(q, k.transpose(-2, -1)) / self.scale

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)

        # (B, heads, N, dim_head) -> (B, N, dim)
        out = rearrange(out, "b h n d -> b n (h d)", h=self.heads, d=self.dim_head)

        return self.to_out(out)


class FFN(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=dim, out_features=hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=hidden_dim, out_features=dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, dim, num_heads, dim_head, mlp_ratio=4.0, dropout=0):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads, dim_head, dropout)
        self.mlp = FFN(dim, int(dim * mlp_ratio), dropout)

    def forward(self, x):
        x = x + self.attn(self.norm(x))
        x = x + self.mlp(self.norm(x))

        return x


class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()

        image_height, image_width = image_size
        patch_height, patch_width = patch_size

        assert image_height % patch_height == 0 and image_width % patch_width == 0, "パッチサイズは，入力画像のサイズを割り切れる必要があります．"

        num_patches = (image_height // patch_height) * (image_width // patch_width)  # パッチの数
        patch_dim = in_channels * patch_height * patch_width  # 各パッチを平坦化したときの次元数

        self.to_patch_embedding = nn.Sequential(
            Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=patch_height, p2=patch_width),  # 画像をパッチに分割して平坦化
            nn.Linear(in_features=patch_dim, out_features=embed_dim),  # 埋め込みを行う
        )

    def forward(self, x):
        """
        B: バッチサイズ
        C: 入力画像のチャネル数
        H: 入力画像の高さ
        W: 入力画像の幅
        """
        return self.to_patch_embedding(x)


class PatchShuffle(nn.Module):
    def __init__(self, ratio):
        # ratio: Encoderに入力しないパッチの割合
        super().__init__()
        self.ratio = ratio

    def forward(self, patches):
        """
        B: バッチサイズ
        N: 系列長（＝パッチの数）
        dim: 次元数（＝埋め込みの次元数）
        """
        B, N, dim = patches.shape
        remain_N = int(N * (1 - self.ratio))  # Encoderに入力するパッチの数

        indexes = [random_indexes(N) for _ in range(B)]  # バッチごとに異なる順番のindexを作る
        forward_indexes = torch.as_tensor(np.stack([i[0] for i in indexes], axis=-1), dtype=torch.long).T.to(patches.device)  # バッチを並べ替えるときのidx (B, N)
        backward_indexes = torch.as_tensor(np.stack([i[1] for i in indexes], axis=-1), dtype=torch.long).T.to(patches.device)  # 並べ替えたパッチをもとの順番に戻すためのidx  (B, N)

        patches = take_indexes(patches, forward_indexes)  # パッチを並べ替える
        patches = patches[:, :remain_N, :]  # Encoderに入力するパッチを抽出

        return patches, forward_indexes, backward_indexes


class MAE_Encoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=192, depth=8, heads=6, dim_head=32, mlp_ratio=4.0, mask_ratio=0.75, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.mask_ratio = mask_ratio


        num_patches = (img_size[0] // patch_size[0]) * (img_size[1] // patch_size[1])
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim) * 0.02)
        self.patch_shuffle = PatchShuffle(mask_ratio)

        self.blocks = nn.ModuleList([
            Block(embed_dim, heads, dim_head, mlp_ratio, dropout)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed

        x, forward_indexes, backward_indexes = self.patch_shuffle(x)

        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        return x, forward_indexes, backward_indexes


class MAE_Decoder(nn.Module):
    def __init__(self, num_patches=64, embed_dim=192, decoder_embed_dim=128, decoder_depth=4, decoder_heads=4, decoder_dim_head=32, mlp_ratio=4.0, patch_size=4, dropout=0.1):
        super().__init__()
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.randn(decoder_embed_dim) * 0.02)
        self.decoder_pos_embed = nn.Parameter(torch.randn(1, num_patches, decoder_embed_dim) * 0.02)

        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_heads, decoder_dim_head, mlp_ratio, dropout)
            for _ in range(decoder_depth)
        ])

        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * 3)

    def forward(self, x, backward_indexes):
        x = self.decoder_embed(x)

        mask_tokens = self.mask_token.repeat(x.shape[0], backward_indexes.shape[1] - x.shape[1], 1)
        x = torch.cat([x, mask_tokens], dim=1)

        x = take_indexes(x, backward_indexes)
        x = x + self.decoder_pos_embed

        for block in self.decoder_blocks:
            x = block(x)
        x = self.decoder_norm(x)

        x = self.decoder_pred(x)

        return x


class MAE_ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=192, depth=8, heads=6, dim_head=32, decoder_embed_dim=128, decoder_depth=4, decoder_heads=4, decoder_dim_head=32, mlp_ratio=4.0, mask_ratio=0.75, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size

        self.encoder = MAE_Encoder(
            (img_size, img_size), (patch_size, patch_size), embed_dim, depth, heads, dim_head, mlp_ratio, mask_ratio, dropout
        )

        self.decoder = MAE_Decoder(
            (img_size // patch_size) ** 2, embed_dim, decoder_embed_dim, decoder_depth, decoder_heads, decoder_dim_head, mlp_ratio, patch_size, dropout
        )

    def patchify(self, imgs):
        p = self.patch_size
        assert imgs.shape[2] == imgs.shape[3] and imgs.shape[2] % p == 0

        h = w = imgs.shape[2] // p
        x = imgs.reshape(imgs.shape[0], 3, h, p, w, p)
        x = x.permute(0, 2, 4, 3, 5, 1).reshape(imgs.shape[0], h * w, p**2 * 3)
        x = x.reshape(imgs.shape[0], h * w, p**2 * 3)
        return x

    def forward(self, imgs):
        latent, forward_indexes, backward_indexes = self.encoder(imgs)

        pred = self.decoder(latent, backward_indexes)

        return pred, forward_indexes, backward_indexes, latent

    def compute_loss(self, imgs, pred, forward_indexes, backward_indexes, latent):
        target = self.patchify(imgs)

        B, N = backward_indexes.shape
        remain_N = pred.shape[1] - (N - latent.shape[1])

        mask = torch.ones(B, N, device=imgs.device)

        for b in range(B):
            mask[b, forward_indexes[b, :remain_N]] = 0

        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)

        loss = (loss * mask).sum() / mask.sum()

        return loss

### 事前学習（自己教師あり学習）

In [14]:
class Classifier(nn.Module):
    def __init__(self, encoder, embed_dim=192, num_classes=10, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        hidden_dim = 128

        for param in self.encoder.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
      features, _, _, _ = self.encoder(x)
      features = features.mean(dim=1)

      logits = self.classifier(features)
      return logits


In [7]:
def suggest_mae_params(trial):
    return{
        'embed_dim': trial.suggest_categorical('embed_dim', [128, 192, 256]),
        'depth': trial.suggest_int('depth', 6, 12),
        'heads': trial.suggest_categorical('heads', [4, 6, 8]),
        'decoder_embed_dim': trial.suggest_categorical('decoder_embed_dim', [96, 128, 192]),
        'decoder_depth': trial.suggest_int('decoder_depth', 2, 6),
        'decoder_heads': trial.suggest_categorical('decoder_heads', [4, 6, 8]),
        'mask_ratio': trial.suggest_float('mask_ratio', 0.5, 0.8),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'lr': trial.suggest_float('lr', 1e-5, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'warmup_epochs': trial.suggest_int('warmup_epochs', 1, 4),
    }

def suggest_classifier_params(trial):
    return{
        'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
    }

In [8]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr_ratio=0.01):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr_ratio = min_lr_ratio
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            # Warmup phase
            lr_scale = epoch / self.warmup_epochs
        else:
            # Cosine annealing phase
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr_scale = self.min_lr_ratio + (1 - self.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

        for param_group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            param_group['lr'] = base_lr * lr_scale

def save_checkpoint(model, optimizer, scheduler, epoch, loss, filepath):
    """Save training checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    if scheduler is not None and hasattr(scheduler, 'state_dict'):
        checkpoint['scheduler_state_dict'] = scheduler.state_dict()

    torch.save(checkpoint, filepath)

def load_checkpoint(model, optimizer, scheduler, filepath, device):
    """Load training checkpoint"""
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in checkpoint and hasattr(scheduler, 'load_state_dict'):
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    return checkpoint['epoch'], checkpoint['loss']

In [9]:
class EarlyStopping:
    def __init__(self, patience=5, mode='min', delta=0.0):
        """
        mode: 'min'（lossなど） or 'max'（accuracyなど）
        """
        self.patience = patience
        self.mode = mode
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, metric):
        score = -metric if self.mode == 'min' else metric

        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0


In [15]:
def train_mae_model(model, dataloader_train, dataloader_valid, device,
                   epochs, optimizer, scheduler=None, save_interval=2,
                   checkpoint_dir=work_dir+'/Lecture10/checkpoints', patience=5, start_epoch=0):
    """Train MAE model with checkpointing"""

    os.makedirs(os.path.dirname(resume_path), exist_ok=True)

    model.train()

    train_losses = []
    val_losses = []

    early_stopper = EarlyStopping(patience=patience)

    for epoch in range(start_epoch, epochs):
        # Training phase
        model.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, (images, _) in enumerate(tqdm(dataloader_train, desc=f'Epoch {epoch+1}/{epochs}')):
            images = images.to(device)

            optimizer.zero_grad()

            # Forward pass
            pred, ids_restore, ids_keep, latent = model(images)
            loss = model.compute_loss(images, pred, ids_restore, ids_keep, latent)

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_loss / num_batches
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()
        val_loss = 0
        val_batches = 0

        with torch.no_grad():
            for images, _ in dataloader_valid:
                images = images.to(device)
                pred, ids_restore, ids_keep, latent = model(images)
                loss = model.compute_loss(images, pred, ids_restore, ids_keep, latent)
                val_loss += loss.item()
                val_batches += 1

        avg_val_loss = val_loss / val_batches
        val_losses.append(avg_val_loss)

        # Update learning rate
        if scheduler:
            scheduler.step(epoch)

        print(f'Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}')

        # Save checkpoint
        if (epoch + 1) % save_interval == 0:
            save_checkpoint(
                model, optimizer, scheduler, epoch, avg_val_loss,
                f'{checkpoint_dir}/mae_checkpoint_epoch_{epoch+1}.pth'
            )

        early_stopper.step(avg_val_loss)

        if early_stopper.early_stop:
            print(f'Early stopping at epoch {epoch+1}')
            break

    return train_losses, val_losses

def train_classifier(model, dataloader_train, dataloader_valid, device,
                    epochs, optimizer, scheduler=None, patience=5):
    """Train classifier with frozen encoder"""

    model.train()
    best_val_acc = 0

    early_stopper = EarlyStopping(patience=patience, mode='max')

    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in tqdm(dataloader_train, desc=f'Epoch {epoch+1}/{epochs}'):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total

        # Validation phase
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for images, labels in dataloader_valid:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                loss = F.cross_entropy(logits, labels)

                val_loss += loss.item()
                _, predicted = torch.max(logits.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if scheduler:
            scheduler.step()

        print(f'Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        early_stopper.step(val_acc)
        if early_stopper.early_stop:
            print(f'Early stopping at epoch {epoch+1}')
            break

    return best_val_acc

In [20]:
def get_recommended_config():
    mae_config = {
        'embed_dim': 192,
        'depth': 8,
        'heads': 6,
        'decoder_embed_dim': 128,
        'decoder_depth': 4,
        'decoder_heads': 4,
        'mask_ratio': 0.75,
        'dropout': 0.1,
        'lr': 1e-4,
        'batch_size': 128,
        'warmup_epochs': 10,
        'start_epoch': 75,
        'total_epochs': 100
    }

    classifier_config = {
        'lr': 1e-3,
        'batch_size': 128,
        'dropout': 0.2,
        'weight_decay': 1e-4,
        'epochs': 20
    }

    return mae_config, classifier_config

def train_with_recommended_config(dataloader_train, dataloader_valid, device):
    """Train model with recommended configuration"""

    resume_path = work_dir + '/Lecture10/checkpoints/mae_checkpoint_epoch_74.pth'

    mae_config, classifier_config = get_recommended_config()

    # Phase 1: Train MAE
    print("Training MAE with recommended configuration...")
    model = MAE_ViT(
        img_size=32,
        patch_size=4,
        embed_dim=mae_config['embed_dim'],
        depth=mae_config['depth'],
        heads=mae_config['heads'],
        dim_head=32,
        decoder_embed_dim=mae_config['decoder_embed_dim'],
        decoder_depth=mae_config['decoder_depth'],
        decoder_heads=mae_config['decoder_heads'],
        decoder_dim_head=32,
        mlp_ratio=4.0,
        mask_ratio=mae_config['mask_ratio'],
        dropout=mae_config['dropout']
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                 lr=mae_config['lr'], weight_decay=0.05)
    scheduler = WarmupCosineScheduler(optimizer, mae_config['warmup_epochs'],
                                    mae_config['total_epochs'], min_lr_ratio=0.001)
    if os.path.exists(resume_path):
        print("Resuming from checkpoint:", resume_path)
        start_epoch, last_loss = load_checkpoint(model, optimizer, scheduler, resume_path, device)

    train_mae_model(model, dataloader_train, dataloader_valid, device,
                   epochs=mae_config['total_epochs'], optimizer=optimizer, scheduler=scheduler, save_interval=2, checkpoint_dir=resume_path, patience=5, start_epoch=mae_config['start_epoch'])

    # Phase 2: Train classifier
    print("Training classifier...")

    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    _, _, _, latent = model(dummy_input)
    actual_embed_dim = latent.shape[-1]
    classifier = Classifier(
        encoder=model,
        embed_dim=actual_embed_dim,
        num_classes=10,
        dropout=classifier_config['dropout']
    ).to(device)

    optimizer = torch.optim.AdamW(
        classifier.classifier.parameters(),
        lr=classifier_config['lr'],
        weight_decay=classifier_config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=classifier_config['epochs'])

    best_val_acc = train_classifier(classifier, dataloader_train, dataloader_valid, device,
                                  epochs=classifier_config['epochs'], optimizer=optimizer, scheduler=scheduler)

    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    return classifier

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = train_with_recommended_config(dataloader_train, dataloader_valid, device)

Training MAE with recommended configuration...
Resuming from checkpoint: drive/MyDrive/DLBasic/HW/HW10/Lecture10/checkpoints/mae_checkpoint_epoch_34.pth


Epoch 36/100:   0%|          | 0/368 [00:00<?, ?it/s]

Epoch 36: Train Loss = 0.0677, Val Loss = 0.0803


Epoch 37/100:   0%|          | 0/368 [00:00<?, ?it/s]

Epoch 37: Train Loss = 0.0675, Val Loss = 0.0797


...

Epoch 73: Train Loss = 0.0618, Val Loss = 0.0746


Epoch 74/100:   0%|          | 0/368 [00:00<?, ?it/s]

Epoch 74: Train Loss = 0.0619, Val Loss = 0.0740
Early stopping at epoch 74
Training classifier...


Epoch 1/20:   0%|          | 0/368 [00:00<?, ?it/s]

RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x48 and 192x10)

In [ ]:
save = 'y'

In [ ]:
if save == 'y':
    model.eval()

    t_pred = []
    for x in dataloader_test:
        x = x.to(device)
        y = model(x)

        # モデルの出力を予測値のスカラーに変換
        pred = y.argmax(1).tolist()
        t_pred.extend(pred)

    submission = pd.Series(t_pred, name='label')
    submission.to_csv(work_dir + '/Lecture10/submission_pred.csv', header=True, index_label='id')